# Gold — Perfil socioeconômico de clientes por UF

Desenvolvido por: Ygor Moraes

## Objetivo

Criar a Gold `gold_ecommerce_clientes_perfil_socioeconomico_uf`, enriquecendo clientes com renda média per capita da UF do endereço principal.

## Regra de negócio

Cada cliente deve ser associado ao estado do seu endereço principal.

Depois, o estado é cruzado com a Silver `ibge_renda_uf` para trazer a renda média per capita da UF.

A Gold calcula:

- quantidade de clientes por UF;
- percentual de clientes por UF;
- quantidade de clientes com match na base IBGE;
- quantidade de clientes sem match na base IBGE;
- percentual de match IBGE;
- renda média per capita da UF;
- ano e fonte da referência IBGE.

## Fontes

- Silver `ecommerce_clientes`
- Silver `ecommerce_enderecos`
- Silver `ibge_renda_uf`

## Cuidados técnicos

- `ecommerce_clientes` é lida como Delta.
- `ecommerce_enderecos` é lida como Delta.
- `ibge_renda_uf` é lida como Delta.
- Endereços são deduplicados por `id_endereco`.
- Apenas o endereço principal entra na regra.
- O join deve manter uma linha por cliente.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Importa funções e define parâmetros da Gold.

from pyspark.sql.functions import (
    col,
    countDistinct,
    current_timestamp,
    lit,
    lower,
    max as spark_max,
    regexp_replace,
    round as spark_round,
    sum as spark_sum,
    to_timestamp,
    trim,
    upper,
    when,
    row_number
)

from pyspark.sql.window import Window

SILVER_CLIENTES_TABLE = "ecommerce_clientes"
SILVER_ENDERECOS_TABLE = "ecommerce_enderecos"
SILVER_IBGE_RENDA_UF_TABLE = "ibge_renda_uf"

SILVER_CLIENTES_PATH = f"{SILVER_BASE_PATH}{SILVER_CLIENTES_TABLE}"
SILVER_ENDERECOS_PATH = f"{SILVER_BASE_PATH}{SILVER_ENDERECOS_TABLE}"
SILVER_IBGE_RENDA_UF_PATH = f"{SILVER_BASE_PATH}{SILVER_IBGE_RENDA_UF_TABLE}"

GOLD_TABLE = "gold_ecommerce_clientes_perfil_socioeconomico_uf"
GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_TABLE}"

FINAL_TABLE = f"{TARGET_SCHEMA}.{GOLD_TABLE}"

CLIENTES_REQUIRED_COLUMNS = [
    "id_cliente"
]

ENDERECOS_REQUIRED_COLUMNS = [
    "id_endereco",
    "id_cliente",
    "cep",
    "estado",
    "cidade",
    "is_principal",
    "silver_processed_at"
]

IBGE_REQUIRED_COLUMNS = [
    "uf",
    "nome_uf",
    "renda_media_per_capita",
    "ano_referencia",
    "fonte"
]

GOLD_KEY_COLUMNS = [
    "estado"
]

adls_options = get_adls_options()

print("Parâmetros definidos com sucesso.")
print("SILVER_CLIENTES_PATH:", SILVER_CLIENTES_PATH)
print("SILVER_ENDERECOS_PATH:", SILVER_ENDERECOS_PATH)
print("SILVER_IBGE_RENDA_UF_PATH:", SILVER_IBGE_RENDA_UF_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("FINAL_TABLE:", FINAL_TABLE)

In [0]:
# Lê as Silvers necessárias para montar a Gold.

df_clientes = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_CLIENTES_PATH)
)

df_enderecos = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_ENDERECOS_PATH)
)

df_ibge_renda_uf = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_IBGE_RENDA_UF_PATH)
)

total_clientes = df_clientes.count()
total_enderecos = df_enderecos.count()
total_ufs_ibge = df_ibge_renda_uf.count()

print("Silver de clientes lida com sucesso.")
print(f"Total clientes: {total_clientes}")

print("Silver de endereços lida com sucesso em Delta.")
print(f"Total endereços: {total_enderecos}")

print("Silver IBGE renda UF lida com sucesso.")
print(f"Total UFs IBGE: {total_ufs_ibge}")

In [0]:
# Valida colunas obrigatórias e chaves principais das fontes.

validate_required_columns(df_clientes, CLIENTES_REQUIRED_COLUMNS)
validate_required_columns(df_enderecos, ENDERECOS_REQUIRED_COLUMNS)
validate_required_columns(df_ibge_renda_uf, IBGE_REQUIRED_COLUMNS)

clientes_distintos = (
    df_clientes
    .select(col("id_cliente").cast("int").alias("id_cliente"))
    .distinct()
    .count()
)

enderecos_distintos = (
    df_enderecos
    .select(col("id_endereco").cast("int").alias("id_endereco"))
    .distinct()
    .count()
)

ufs_ibge_distintas = (
    df_ibge_renda_uf
    .select(upper(trim(col("uf"))).alias("uf"))
    .distinct()
    .count()
)

clientes_duplicados = total_clientes - clientes_distintos
enderecos_duplicados = total_enderecos - enderecos_distintos
ufs_ibge_duplicadas = total_ufs_ibge - ufs_ibge_distintas

print(f"Total clientes: {total_clientes}")
print(f"Clientes distintos: {clientes_distintos}")
print(f"Clientes duplicados: {clientes_duplicados}")

print(f"Total endereços: {total_enderecos}")
print(f"Endereços distintos: {enderecos_distintos}")
print(f"Endereços duplicados por id_endereco: {enderecos_duplicados}")

print(f"Total UFs IBGE: {total_ufs_ibge}")
print(f"UFs distintas IBGE: {ufs_ibge_distintas}")
print(f"UFs duplicadas IBGE: {ufs_ibge_duplicadas}")

if clientes_duplicados > 0:
    raise Exception("Erro: existem clientes duplicados por id_cliente.")

if enderecos_distintos == 0:
    raise Exception("Erro: não foram encontrados endereços válidos por id_endereco.")

if ufs_ibge_duplicadas > 0:
    raise Exception("Erro: existem UFs duplicadas na Silver IBGE.")

print("Validação OK: fontes mínimas conferidas.")

In [0]:
# Deduplica endereços por id_endereco mantendo o registro mais recente.

df_enderecos_base = (
    df_enderecos
    .withColumn("id_endereco_int", col("id_endereco").cast("int"))
    .withColumn("id_cliente_int", col("id_cliente").cast("int"))
    .withColumn("silver_processed_at_ts", to_timestamp(col("silver_processed_at")))
)

window_enderecos_dedup = (
    Window
    .partitionBy("id_endereco_int")
    .orderBy(
        col("silver_processed_at_ts").desc_nulls_last()
    )
)

df_enderecos_dedup = (
    df_enderecos_base
    .withColumn("rn", row_number().over(window_enderecos_dedup))
    .filter(col("rn") == 1)
    .drop("rn")
)

print("Endereços deduplicados por id_endereco.")

In [0]:
# Valida se a deduplicação deixou apenas um registro por endereço.

total_enderecos_dedup = df_enderecos_dedup.count()

enderecos_distintos_dedup = (
    df_enderecos_dedup
    .select("id_endereco_int")
    .distinct()
    .count()
)

enderecos_duplicados_dedup = total_enderecos_dedup - enderecos_distintos_dedup

enderecos_id_nulo = (
    df_enderecos_dedup
    .filter(col("id_endereco_int").isNull())
    .count()
)

clientes_id_nulo_em_enderecos = (
    df_enderecos_dedup
    .filter(col("id_cliente_int").isNull())
    .count()
)

print(f"Total endereços original: {total_enderecos}")
print(f"Total endereços após deduplicação: {total_enderecos_dedup}")
print(f"Endereços distintos após deduplicação: {enderecos_distintos_dedup}")
print(f"Endereços duplicados restantes: {enderecos_duplicados_dedup}")
print(f"Endereços com id_endereco nulo: {enderecos_id_nulo}")
print(f"Endereços com id_cliente nulo: {clientes_id_nulo_em_enderecos}")

if enderecos_duplicados_dedup > 0:
    raise Exception("Erro: ainda existem endereços duplicados por id_endereco.")

if enderecos_id_nulo > 0:
    raise Exception("Erro: existem endereços com id_endereco nulo.")

print("Validação OK: endereços deduplicados corretamente.")

In [0]:
# Seleciona o endereço principal de cada cliente.

df_enderecos_principais = (
    df_enderecos_dedup
    .withColumn(
        "is_principal_normalizado",
        lower(trim(col("is_principal").cast("string")))
    )
    .filter(
        col("is_principal_normalizado").isin("true", "1", "sim", "s")
    )
    .select(
        col("id_cliente_int").alias("id_cliente"),
        upper(trim(col("estado"))).alias("estado")
    )
)

total_enderecos_principais = df_enderecos_principais.count()

clientes_com_endereco_principal = (
    df_enderecos_principais
    .select("id_cliente")
    .distinct()
    .count()
)

clientes_duplicados_endereco_principal = (
    total_enderecos_principais - clientes_com_endereco_principal
)

enderecos_principais_sem_estado = (
    df_enderecos_principais
    .filter(col("estado").isNull() | (col("estado") == ""))
    .count()
)

print(f"Total endereços principais: {total_enderecos_principais}")
print(f"Clientes com endereço principal: {clientes_com_endereco_principal}")
print(f"Clientes duplicados com endereço principal: {clientes_duplicados_endereco_principal}")
print(f"Endereços principais sem estado: {enderecos_principais_sem_estado}")

if clientes_duplicados_endereco_principal > 0:
    raise Exception("Erro: existe mais de um endereço principal por cliente.")

if enderecos_principais_sem_estado > 0:
    raise Exception("Erro: existem endereços principais sem estado.")

print("Validação OK: endereço principal por cliente conferido.")

In [0]:
# Prepara a base IBGE para join por UF.

df_ibge_renda_uf_base = (
    df_ibge_renda_uf
    .select(
        upper(trim(col("uf"))).alias("estado"),
        trim(col("nome_uf")).alias("nome_uf"),
        col("renda_media_per_capita").cast("decimal(18,2)").alias("renda_media_per_capita"),
        col("ano_referencia").cast("int").alias("ano_referencia"),
        trim(col("fonte")).alias("fonte")
    )
)

total_ibge_base = df_ibge_renda_uf_base.count()

ufs_ibge_base = (
    df_ibge_renda_uf_base
    .select("estado")
    .distinct()
    .count()
)

ufs_ibge_duplicadas_base = total_ibge_base - ufs_ibge_base

ufs_ibge_com_renda_nula = (
    df_ibge_renda_uf_base
    .filter(col("renda_media_per_capita").isNull())
    .count()
)

print(f"Total linhas IBGE base: {total_ibge_base}")
print(f"UFs distintas IBGE base: {ufs_ibge_base}")
print(f"UFs duplicadas IBGE base: {ufs_ibge_duplicadas_base}")
print(f"UFs com renda nula: {ufs_ibge_com_renda_nula}")

if ufs_ibge_duplicadas_base > 0:
    raise Exception("Erro: existem UFs duplicadas na base IBGE preparada.")

if ufs_ibge_com_renda_nula > 0:
    raise Exception("Erro: existem UFs com renda média nula na base IBGE.")

print("Validação OK: base IBGE preparada para join.")

In [0]:
# Junta clientes com endereço principal e renda IBGE por UF.

df_clientes_ibge_base = (
    df_clientes
    .select(
        col("id_cliente").cast("int").alias("id_cliente")
    )
    .join(
        df_enderecos_principais,
        on="id_cliente",
        how="left"
    )
    .join(
        df_ibge_renda_uf_base,
        on="estado",
        how="left"
    )
    .withColumn(
        "fl_match_ibge",
        when(col("renda_media_per_capita").isNotNull(), 1).otherwise(0)
    )
)

print("Base de clientes com UF e IBGE criada.")

In [0]:
# Valida se os joins mantiveram um registro por cliente.

total_clientes_ibge_base = df_clientes_ibge_base.count()

clientes_distintos_ibge_base = (
    df_clientes_ibge_base
    .select("id_cliente")
    .distinct()
    .count()
)

clientes_duplicados_ibge_base = total_clientes_ibge_base - clientes_distintos_ibge_base

clientes_sem_estado = (
    df_clientes_ibge_base
    .filter(col("estado").isNull() | (col("estado") == ""))
    .count()
)

clientes_sem_match_ibge = (
    df_clientes_ibge_base
    .filter(col("fl_match_ibge") == 0)
    .count()
)

print(f"Total clientes original: {total_clientes}")
print(f"Total clientes após joins: {total_clientes_ibge_base}")
print(f"Clientes distintos após joins: {clientes_distintos_ibge_base}")
print(f"Clientes duplicados após joins: {clientes_duplicados_ibge_base}")
print(f"Clientes sem estado: {clientes_sem_estado}")
print(f"Clientes sem match IBGE: {clientes_sem_match_ibge}")

if total_clientes_ibge_base != total_clientes:
    raise Exception("Erro: os joins alteraram a quantidade de clientes.")

if clientes_duplicados_ibge_base > 0:
    raise Exception("Erro: os joins geraram clientes duplicados.")

if clientes_sem_estado > 0:
    raise Exception("Erro: existem clientes sem estado após o join.")

print("Validação OK: joins mantiveram 1 registro por cliente.")

In [0]:
# Cria a Gold de perfil socioeconômico por UF.

df_gold = (
    df_clientes_ibge_base
    .groupBy(
        "estado",
        "nome_uf",
        "renda_media_per_capita",
        "ano_referencia",
        "fonte"
    )
    .agg(
        countDistinct("id_cliente").alias("qtd_clientes"),
        spark_sum("fl_match_ibge").alias("qtd_clientes_com_match_ibge")
    )
    .withColumn(
        "qtd_clientes_sem_match_ibge",
        col("qtd_clientes") - col("qtd_clientes_com_match_ibge")
    )
    .withColumn(
        "percentual_clientes",
        spark_round((col("qtd_clientes") / lit(total_clientes)) * 100, 2)
    )
    .withColumn(
        "percentual_match_ibge",
        spark_round((col("qtd_clientes_com_match_ibge") / col("qtd_clientes")) * 100, 2)
    )
    .withColumn("gold_processed_at", current_timestamp())
    .orderBy("estado")
)

print("Gold de perfil socioeconômico por UF criada em memória.")
display(df_gold)

In [0]:
# Valida totais, duplicidade de UF e campos principais da Gold.

total_linhas_gold = df_gold.count()

total_estados_distintos = (
    df_gold
    .select(*GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

estados_duplicados = total_linhas_gold - total_estados_distintos

validacao_gold = (
    df_gold
    .agg(
        spark_sum("qtd_clientes").alias("total_clientes_gold"),
        spark_sum("qtd_clientes_com_match_ibge").alias("total_clientes_com_match_ibge"),
        spark_sum("qtd_clientes_sem_match_ibge").alias("total_clientes_sem_match_ibge")
    )
    .collect()[0]
)

nulos_gold = (
    df_gold
    .filter(
        col("estado").isNull() |
        col("qtd_clientes").isNull() |
        col("percentual_clientes").isNull() |
        col("qtd_clientes_com_match_ibge").isNull() |
        col("qtd_clientes_sem_match_ibge").isNull() |
        col("percentual_match_ibge").isNull()
    )
    .count()
)

print(f"Total clientes fonte: {total_clientes}")
print(f"Total clientes na Gold: {validacao_gold['total_clientes_gold']}")
print(f"Clientes com match IBGE: {validacao_gold['total_clientes_com_match_ibge']}")
print(f"Clientes sem match IBGE: {validacao_gold['total_clientes_sem_match_ibge']}")
print(f"Total linhas Gold: {total_linhas_gold}")
print(f"Estados duplicados: {estados_duplicados}")
print(f"Linhas com nulos principais: {nulos_gold}")

if validacao_gold["total_clientes_gold"] != total_clientes:
    raise Exception("Erro: total de clientes da Gold não fecha com a fonte.")

if (
    validacao_gold["total_clientes_com_match_ibge"] +
    validacao_gold["total_clientes_sem_match_ibge"]
    != validacao_gold["total_clientes_gold"]
):
    raise Exception("Erro: clientes com + sem match IBGE não fecha com total.")

if estados_duplicados > 0:
    raise Exception("Erro: existem estados duplicados na Gold.")

if nulos_gold > 0:
    raise Exception("Erro: existem nulos nas colunas principais da Gold.")

print("Validação OK: Gold em memória conferida.")

In [0]:
# Grava a Gold em Delta no ADLS.

(
    df_gold
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .save(GOLD_PATH)
)

print(f"Gold gravada com sucesso em Delta: {GOLD_PATH}")

In [0]:
# Lê e valida a Gold Delta gravada.

df_gold_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(GOLD_PATH)
)

total_linhas_gold_saved = df_gold_saved.count()

total_estados_saved = (
    df_gold_saved
    .select(*GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

estados_duplicados_saved = total_linhas_gold_saved - total_estados_saved

validacao_gold_saved = (
    df_gold_saved
    .agg(
        spark_sum("qtd_clientes").alias("total_clientes_gold"),
        spark_sum("qtd_clientes_com_match_ibge").alias("total_clientes_com_match_ibge"),
        spark_sum("qtd_clientes_sem_match_ibge").alias("total_clientes_sem_match_ibge")
    )
    .collect()[0]
)

nulos_gold_saved = (
    df_gold_saved
    .filter(
        col("estado").isNull() |
        col("qtd_clientes").isNull() |
        col("percentual_clientes").isNull() |
        col("qtd_clientes_com_match_ibge").isNull() |
        col("qtd_clientes_sem_match_ibge").isNull() |
        col("percentual_match_ibge").isNull()
    )
    .count()
)

print(f"Total linhas Gold Delta: {total_linhas_gold_saved}")
print(f"Estados duplicados Gold Delta: {estados_duplicados_saved}")
print(f"Total clientes fonte: {total_clientes}")
print(f"Total clientes Gold Delta: {validacao_gold_saved['total_clientes_gold']}")
print(f"Clientes com match IBGE Gold Delta: {validacao_gold_saved['total_clientes_com_match_ibge']}")
print(f"Clientes sem match IBGE Gold Delta: {validacao_gold_saved['total_clientes_sem_match_ibge']}")
print(f"Linhas com nulos principais Gold Delta: {nulos_gold_saved}")

if validacao_gold_saved["total_clientes_gold"] != total_clientes:
    raise Exception("Erro: total de clientes da Gold Delta não confere.")

if (
    validacao_gold_saved["total_clientes_com_match_ibge"] +
    validacao_gold_saved["total_clientes_sem_match_ibge"]
    != validacao_gold_saved["total_clientes_gold"]
):
    raise Exception("Erro: clientes com + sem match IBGE não fecha na Gold Delta.")

if estados_duplicados_saved > 0:
    raise Exception("Erro: existem estados duplicados na Gold Delta.")

if nulos_gold_saved > 0:
    raise Exception("Erro: existem nulos nas colunas principais da Gold Delta.")

print("Validação OK: Gold Delta gravada corretamente.")

In [0]:
# Prepara a Gold para escrita no SQL Server.

df_gold_sql = (
    df_gold_saved
    .select(
        col("estado").cast("string").alias("estado"),
        col("nome_uf").cast("string").alias("nome_uf"),
        col("qtd_clientes").cast("int").alias("qtd_clientes"),
        col("percentual_clientes").cast("decimal(10,2)").alias("percentual_clientes"),
        col("qtd_clientes_com_match_ibge").cast("int").alias("qtd_clientes_com_match_ibge"),
        col("qtd_clientes_sem_match_ibge").cast("int").alias("qtd_clientes_sem_match_ibge"),
        col("percentual_match_ibge").cast("decimal(10,2)").alias("percentual_match_ibge"),
        col("renda_media_per_capita").cast("decimal(18,2)").alias("renda_media_per_capita"),
        col("ano_referencia").cast("int").alias("ano_referencia"),
        col("fonte").cast("string").alias("fonte"),
        col("gold_processed_at").cast("timestamp").alias("gold_processed_at")
    )
)

print("Gold preparada para escrita no SQL Server.")
df_gold_sql.printSchema()
display(df_gold_sql.orderBy("estado"))

In [0]:
# Grava a Gold diretamente na tabela final do SQL Server.

write_sql_table(
    df=df_gold_sql,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    mode="overwrite",
    sql_port=SQL_PORT
)

print(f"Gold gravada com sucesso na tabela final: {FINAL_TABLE}")

In [0]:
# Lê e valida a tabela final do SQL Server.

df_final = read_sql_table(
    spark=spark,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    sql_port=SQL_PORT
)

total_linhas_final = df_final.count()

total_estados_final = (
    df_final
    .select(*GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

estados_duplicados_final = total_linhas_final - total_estados_final

validacao_final = (
    df_final
    .agg(
        spark_sum("qtd_clientes").alias("total_clientes_gold"),
        spark_sum("qtd_clientes_com_match_ibge").alias("total_clientes_com_match_ibge"),
        spark_sum("qtd_clientes_sem_match_ibge").alias("total_clientes_sem_match_ibge")
    )
    .collect()[0]
)

nulos_final = (
    df_final
    .filter(
        col("estado").isNull() |
        col("qtd_clientes").isNull() |
        col("percentual_clientes").isNull() |
        col("qtd_clientes_com_match_ibge").isNull() |
        col("qtd_clientes_sem_match_ibge").isNull() |
        col("percentual_match_ibge").isNull()
    )
    .count()
)

print(f"Total linhas tabela final: {total_linhas_final}")
print(f"Estados duplicados tabela final: {estados_duplicados_final}")
print(f"Total clientes fonte: {total_clientes}")
print(f"Total clientes tabela final: {validacao_final['total_clientes_gold']}")
print(f"Clientes com match IBGE tabela final: {validacao_final['total_clientes_com_match_ibge']}")
print(f"Clientes sem match IBGE tabela final: {validacao_final['total_clientes_sem_match_ibge']}")
print(f"Linhas com nulos principais tabela final: {nulos_final}")

if validacao_final["total_clientes_gold"] != total_clientes:
    raise Exception("Erro: total de clientes da tabela final não confere.")

if (
    validacao_final["total_clientes_com_match_ibge"] +
    validacao_final["total_clientes_sem_match_ibge"]
    != validacao_final["total_clientes_gold"]
):
    raise Exception("Erro: clientes com + sem match IBGE não fecha na tabela final.")

if estados_duplicados_final > 0:
    raise Exception("Erro: existem estados duplicados na tabela final.")

if nulos_final > 0:
    raise Exception("Erro: existem nulos nas colunas principais da tabela final.")

print("Validação OK: tabela final SQL Server gravada corretamente.")